In [4]:
from dotenv import load_dotenv
load_dotenv()


True

In [1]:
from datasets import load_dataset

In [5]:
# Load the dataset
dataset = load_dataset("b-mc2/sql-create-context", split="train")

print(f"Dataset size: {len(dataset)} examples")
print(f"\nDataset columns: {dataset.column_names}")
print("\n" + "="*60)
print("Sample entry:")
print("="*60)

# Show a sample
sample = dataset[0]
print(f"\nQuestion: {sample['question']}")
print(f"\nContext (Schema):\n{sample['context']}")
print(f"\nAnswer (SQL): {sample['answer']}")

Dataset size: 78577 examples

Dataset columns: ['answer', 'question', 'context']

Sample entry:

Question: How many heads of the departments are older than 56 ?

Context (Schema):
CREATE TABLE head (age INTEGER)

Answer (SQL): SELECT COUNT(*) FROM head WHERE age > 56


In [6]:
import json


def convert_to_messages(example):
    user_message = f"""Given the following database schema:

{example['context']}

Write a SQL query to answer this question: {example['question']}"""
    
    assistant_message = example['answer']
    
    return {
        "messages": [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message}
        ]
    }

# Show an example of the converted format
sample_converted = convert_to_messages(dataset[0])
print("Converted format:")
print(json.dumps(sample_converted, indent=2))

Converted format:
{
  "messages": [
    {
      "role": "user",
      "content": "Given the following database schema:\n\nCREATE TABLE head (age INTEGER)\n\nWrite a SQL query to answer this question: How many heads of the departments are older than 56 ?"
    },
    {
      "role": "assistant",
      "content": "SELECT COUNT(*) FROM head WHERE age > 56"
    }
  ]
}


In [7]:
TRAIN_SIZE = 5000
VAL_SIZE = 100

# Shuffle and select a subset
shuffled_dataset = dataset.shuffle(seed=42)
train_dataset = shuffled_dataset.select(range(min(TRAIN_SIZE, len(dataset))))
val_dataset = shuffled_dataset.select(range(min(VAL_SIZE, len(dataset))))

# Convert to messages format
train_data = [convert_to_messages(example) for example in train_dataset]
val_data = [convert_to_messages(example) for example in val_dataset]

print(f"Training examples: {len(train_data)}")
print(f"Validation examples: {len(val_data)}")

Training examples: 5000
Validation examples: 100


In [8]:
# output dirs
from pathlib import Path
OUTPUT_DIR = Path('./test-run')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# save to jsonl
train_file = OUTPUT_DIR / 'train.jsonl'
val_file = OUTPUT_DIR / 'val.jsonl'

with open(train_file, 'w') as f:
    for item in train_data:
        f.write(json.dumps(item) + '\n')

with open(val_file, 'w') as f:
    for item in val_data:
        f.write(json.dumps(item) + '\n')


print(f"Training data saved to {train_file}")
print(f"Validation data saved to {val_file}")

Training data saved to test-run/train.jsonl
Validation data saved to test-run/val.jsonl


In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

train_kwargs = {
    "model_path": MODEL_NAME,
    "data_path": str(train_file),
    "ckpt_output_dir": OUTPUT_DIR / 'model',
    "dataset_type": "chat_template",
    "field_messages": "messages",
    "num_epochs": 1,
    "learning_rate": 1.5e-4,
    "micro_batch_size": 16,
    "max_seq_len": 1024,
    "seed": 42,
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.1,
    "load_in_4bit": False,
    "bf16": True,
    "sample_packing": False,
    "logging_steps": 10,
    "eval_steps": 10,
    "save_steps": 10,
    "save_total_limit": 1,
    "wandb_project": "sunday-auto-customize",
    "wandb_entity": "ronny21",
    "wandb_run_name": "test-run-v0",
}

In [10]:
from training_hub.algorithms.lora import UnslothLoRABackend, JSONLLoggingCallback

backend = UnslothLoRABackend()

In [11]:
model, tokenizer = backend._load_unsloth_model(train_kwargs)

/workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/training_hub/algorithms/lora.py:167: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 8. Max memory: 79.097 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

unsloth/Qwen3-4B-Instruct-2507 does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [12]:
model = backend._apply_lora_config(model, train_kwargs)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.4 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [13]:
train_dataset = backend._prepare_dataset(train_kwargs, tokenizer)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [14]:
validation_kwargs = dict(train_kwargs)
validation_kwargs["data_path"] = str(val_file)
validation_dataset = backend._prepare_dataset(validation_kwargs, tokenizer)


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [15]:
train_kwargs['data_path'], validation_kwargs['data_path']

('test-run/train.jsonl', 'test-run/val.jsonl')

In [16]:
training_args = backend._build_training_args(train_kwargs)
training_args.do_eval = True
training_args.eval_strategy = "steps"
training_args

UnslothSFTConfig(output_dir='test-run/model', per_device_train_batch_size=16, num_train_epochs=1, max_steps=-1, learning_rate=0.00015, lr_scheduler_type=<SchedulerType.LINEAR: 'linear'>, lr_scheduler_kwargs=None, warmup_steps=10, optim=<OptimizerNames.ADAMW_8BIT: 'adamw_8bit'>, optim_args=None, weight_decay=0.01, adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-08, optim_target_modules=None, gradient_accumulation_steps=1, average_tokens_across_devices=True, max_grad_norm=0.3, label_smoothing_factor=0.0, bf16=True, fp16=False, bf16_full_eval=False, fp16_full_eval=False, tf32=None, gradient_checkpointing=True, gradient_checkpointing_kwargs=None, torch_compile=False, torch_compile_backend=None, torch_compile_mode=None, use_liger_kernel=False, liger_kernel_config=None, use_cache=False, neftune_noise_alpha=None, torch_empty_cache_steps=250, auto_find_batch_size=False, logging_strategy=<IntervalStrategy.STEPS: 'steps'>, logging_steps=10, logging_first_step=False, log_on_each_node=True, logg

In [17]:
training_args.per_device_eval_batch_size = 1

In [18]:
from trl import SFTTrainer

In [19]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    callbacks=[JSONLLoggingCallback(train_kwargs["ckpt_output_dir"])],
)
trainer

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/5000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/100 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [20]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 313
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 1 x 1) = 16
 "-____-"     Trainable parameters = 16,515,072 of 4,038,983,168 (0.41% trained)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: ronny21 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


CommError: Error uploading run: returned error 404: {"data":{"upsertBucket":null},"errors":[{"message":"entity rawhad not found during upsertBucket","path":["upsertBucket"]}]}